# Connectivity Hyperalignment (CHA)

Connectivity Hyperalignment (CHA) aligns subject-specific connectivity profiles into a shared representational space. The current workflow follows the execution logic in `fmriha/pipeline/gui_run.py`: response data are prepared first, functional connectivity (FC) matrices are built from those response data, and the usual hyperalignment stages are then run on the connectivity modality.

The examples assume that response arrays have already been prepared as `.npy` files with shape `timepoints x features` under the HA work directory. CHA-specific FC files are saved under each subject folder as `<seed_step>/connectivity/<seed_structure>`. Each FC matrix has shape `n_target x n_seed`, where each column is the connectivity profile of one seed feature.

The complete parameter tables for `nb.sls`, `gensls.downsample_cifti_subcortical`, `gensls.generate_searchlight_vol`, `ha.cspace_sls`, `ha.xform_sls_con`, and `ha.align_pipe` are provided in `rha.ipynb`. This notebook lists the CHA-specific call differences for those shared functions and gives full parameter tables for CHA-specific additions. The local execution path implemented by the current GUI runner is covered here.

## Scripts Pipeline

The script workflow contains five stages. The first stage prepares three kinds of searchlights for CHA: alignment searchlights, seed definitions, and target definitions. The second stage builds FC files through `ha.fc_build`. The last three stages reuse the RHA hyperalignment functions with `modality="connectivity"` for template and transform estimation.

**Required script inputs**

| Item | Meaning |
|---|---|
| `work_dir` | Root HA work directory. It contains subject folders, `logs`, `masks`, FC outputs, common-space outputs, transform outputs, and aligned outputs. |
| `json_path` | Processing-record JSON path, usually `work_dir / "logs" / "prep_log.json"`. |
| `sub_list` | Subject folder names used for FC construction, common-space construction, transformation calculation, and alignment. |
| Prepared response arrays | `.npy` response files selected with BIDS-like filters. Common filters include `run`, `ses`, `set`, `space`, `hemi`, `roi`, `res`, `desc`, and `task`. |
| CHA alignment searchlights | Searchlights used by `ha.cspace_sls` and `ha.xform_sls_con` after FC files are built. |
| CHA seed definitions | Usually single-feature seeds derived from dense searchlights. These control the `n_seed` axis of each FC matrix. |
| CHA target definitions | Target neighborhoods used to create the `n_target` axis of each FC matrix. Targets may be cortical, subcortical, ROI-based, or mixed across structures. |
| Task names | Labels written into common-space and transform filenames. Use distinct task names for cortical, subcortical, and ROI workflows when running them in the same work directory. |

```python
import os
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

from pathlib import Path

import nibabel as nib
import numpy as np
import neuroboros as nb

from fmriha import gensls, ha
from fmriha.gensls.dsample import downsample_nifti_volume
```

```python
work_dir = Path("/path/to/ha_workdir")
json_path = work_dir / "logs" / "prep_log.json"
sub_list = ["sub-rid000005", "sub-rid000014", "sub-rid000029"]

fit_response_filter = {"run": "01"}
fc_file_filter = {"desc": "fc-zscore", "suffix": "lowall"}
align_source_filter = {"run": "02"}
common_space_filter = {"alg": "procrustes"}

surface_space = "onavg-ico32"
target_surface_space = "onavg-ico8"
radius_surf = 20
target_radius_surf = 13
radius_vol = 4
target_radius_vol = 4

reference_cifti = "/path/to/reference_run01.dtseries.nii"
roi_name = "mPFC"
nifti_mask_path = "/path/to/mPFC_mask.nii.gz"
```

### (1) Searchlight Preparation

CHA uses searchlights in three roles:

| Role | Used by | Typical shape or content |
|---|---|---|
| Alignment searchlights | `ha.cspace_sls`, `ha.xform_sls_con` | Local neighborhoods over the seed feature axis. These match the RHA searchlights. |
| Seed definitions | `ha.fc_build(seed_sls=...)` | Usually one feature per seed, generated from the first element of each dense searchlight. |
| Target definitions | `ha.fc_build(target_sls=...)` | Coarser searchlights or downsampled neighborhoods used to build connectivity profiles. |

Shared searchlight functions are documented in `rha.ipynb`. CHA changes how the returned searchlights are assigned to seed and target roles.

**Cortical**

For cortical CHA, the alignment searchlights use the same `nb.sls` call as RHA, with `mask=True` and `return_dists=True`. CHA then reuses those returned neighborhoods in two additional ways: single-feature seed definitions are built with `[np.array(idx[0]) for idx in sls_surf[lr]]`, and connectivity targets are generated with a lower-resolution `center_space` and `return_dists=False`.

```python
surface_tmp = {
    lr: nb.sls(
        lr,
        radius_surf,
        surface_space,
        mask=True,
        return_dists=True,
    )
    for lr in "lr"
}

sls_surf = {lr: surface_tmp[lr][0] for lr in surface_tmp}
dists_surf = {lr: surface_tmp[lr][1] for lr in surface_tmp}

sls_surf_seed = {
    lr: [np.array(idx[0]) for idx in sls_surf[lr]]
    for lr in "lr"
}

sls_surf_target = {
    lr: nb.sls(
        lr,
        radius=target_radius_surf,
        space=surface_space,
        center_space=target_surface_space,
        mask=True,
        return_dists=False,
    )
    for lr in "lr"
}
```

**Subcortical**

For CIFTI subcortical CHA, dense masks are created with the same `gensls.downsample_cifti_subcortical` settings used for RHA alignment masks: `N=None` and `sls_type="seed"`. Dense volume searchlights are generated with the same `gensls.generate_searchlight_vol` call as RHA, and the first element of each dense searchlight is then used as a single-voxel seed. CHA adds target neighborhoods by calling `gensls.downsample_cifti_subcortical` with `N=1994` and `sls_type="not_seed"` to create target centers, followed by `gensls.generate_searchlight_vol` with dense masks, those center masks, and `threshold=0` for subcortical targets.

```python
subcortical_keys = ["l", "r", "brain_stem"]

mask_dense_subcortical = {
    lr: gensls.downsample_cifti_subcortical(
        cifti_file=reference_cifti,
        N=None,
        mask3d_out=None,
        sls_type="seed",
        voxel_sizes=None,
        drop_all_zero_columns=True,
        whole_mask=False,
        hemi_mask=True,
    )[lr]
    for lr in subcortical_keys
}

sls_info_sub_align = {
    lr: gensls.generate_searchlight_vol(
        mask_dense=mask_dense_subcortical[lr],
        mask_center=None,
        radius=radius_vol,
        threshold=0.2,
        njobs=10,
        verbose=5,
    )
    for lr in subcortical_keys
}

sls_sub_align = {
    f"subcortical-{lr}": sls_info_sub_align[lr]["sls_idx"]
    for lr in subcortical_keys
}
dists_sub_align = {
    f"subcortical-{lr}": sls_info_sub_align[lr]["dists"]
    for lr in subcortical_keys
}

sls_sub_seed = {
    key: np.array([sl[0] for sl in value], dtype=np.int32)
    for key, value in sls_sub_align.items()
}

mask_dense_sub_target = {
    f"subcortical-{lr}": gensls.downsample_cifti_subcortical(
        cifti_file=reference_cifti,
        N=1994,
        mask3d_out=None,
        sls_type="seed",
        voxel_sizes=None,
        drop_all_zero_columns=True,
        whole_mask=False,
        hemi_mask=True,
    )[lr]
    for lr in subcortical_keys
}

mask_center_sub_target = {
    f"subcortical-{lr}": gensls.downsample_cifti_subcortical(
        cifti_file=reference_cifti,
        N=1994,
        mask3d_out=None,
        sls_type="not_seed",
        voxel_sizes=None,
        drop_all_zero_columns=True,
        whole_mask=False,
        hemi_mask=True,
    )[lr]
    for lr in subcortical_keys
}

sls_info_sub_target = {
    key: gensls.generate_searchlight_vol(
        mask_dense=mask_dense_sub_target[key],
        mask_center=mask_center_sub_target[key],
        radius=target_radius_vol,
        threshold=0,
        njobs=10,
        verbose=1,
    )
    for key in mask_dense_sub_target
}

sls_sub_target = {
    key: value["sls_idx"]
    for key, value in sls_info_sub_target.items()
}
```

**Volume ROI**

For a NIfTI ROI, the dense ROI mask is used for alignment searchlights and seed extraction. Target centers are generated by `downsample_nifti_volume`, then passed to `gensls.generate_searchlight_vol`.

Parameters of `downsample_nifti_volume`:

| Parameter | Default | Meaning |
|---|---|---|
| `mask_data` | Required | Boolean or binary 3D ROI mask. Non-zero voxels are candidate target centers. |
| `N` | `None` | Number of target centers to keep. With `None`, the function keeps `ceil(0.062 * n_mask_voxels)`. Values larger than the mask size are clipped to the mask size. |
| `voxel_sizes` | `None` | Optional voxel spacing used when sampling centers. With `None`, voxel-index coordinates are used. |
| `bins_per_axis` | `16` | Number of bins along each axis for near-uniform spatial sampling. |
| `alpha_count` | `0.005` | Weight controlling how strongly bin-count uniformity affects the base selection. |
| `modulus` | `8` | Grid-residue modulus used during candidate selection. |
| `beam_width` | `12` | Number of residue candidates retained during search. |
| `max_offsets` | `64` | Maximum number of residue offsets tested. |
| `rng_seed` | `616` | Random seed used by farthest-point sampling when tie-breaking or growing the selection. |

```python
mask_dense_roi = nib.load(nifti_mask_path).get_fdata().astype(bool)

sls_info_roi_align = gensls.generate_searchlight_vol(
    mask_dense=mask_dense_roi,
    mask_center=None,
    radius=radius_vol,
    threshold=0.2,
    njobs=10,
    verbose=5,
)

sls_roi_align = {f"volume-{roi_name}": sls_info_roi_align["sls_idx"]}
dists_roi_align = {f"volume-{roi_name}": sls_info_roi_align["dists"]}

sls_roi_seed = {
    f"volume-{roi_name}": np.array(
        [sl[0] for sl in sls_info_roi_align["sls_idx"]],
        dtype=np.int32,
    )
}

mask_center_roi_target = downsample_nifti_volume(mask_dense_roi)

sls_info_roi_target = gensls.generate_searchlight_vol(
    mask_dense=mask_dense_roi,
    mask_center=mask_center_roi_target,
    radius=target_radius_vol,
    threshold=0.2,
    njobs=10,
    verbose=5,
)

sls_roi_target = {f"volume-{roi_name}": sls_info_roi_target["sls_idx"]}
```

### (2) Functional Connectivity Calculation

`ha.fc_build` converts prepared response data into CHA connectivity features. For each subject, it loads one seed response file, converts seed searchlights into seed-node time series, loads one target response file per target structure, converts target searchlights into target-node time series, concatenates target nodes across structures, and saves a target-by-seed FC matrix.

Parameters of `ha.fc_build`:

| Parameter | Default | Meaning |
|---|---|---|
| `work_dir` | Required | Root HA work directory. |
| `sub_list` | Required | Subject folders included in FC construction. |
| `seed_step` | Required | Step folder containing seed response files, usually `resample` or `align`. |
| `seed_modality` | Required | Modality folder containing seed data, usually `response`. |
| `seed_structure` | Required | One seed structure folder. The function accepts a string or a one-element list, such as `hemi-L` or `volume-mPFC`. |
| `seed_file_filter` | Required | BIDS-like filters selecting exactly one seed `.npy` file per subject. |
| `seed_sls` | Required | Seed searchlight definitions. In CHA these are usually single-feature seeds. |
| `target_step` | Required | Step folder containing target response files, usually `resample` or `align`. |
| `target_modality` | Required | Modality folder containing target data, usually `response`. |
| `target_structure` | Required | List of target structure folders. Target nodes are concatenated in this order. |
| `target_sls` | Required | Target searchlight definitions matching `target_structure`. |
| `target_file_filter` | Required | BIDS-like filters selecting exactly one target `.npy` file per subject and target structure. |
| `save_modality` | `connectivity` | Output modality folder. The current CHA pipeline uses `connectivity`. |
| `zscore` | `True` | If `True`, z-scores each seed connectivity profile across targets after correlation calculation. |
| `njobs` | `5` | Number of joblib workers. |
| `verbose` | `1` | Joblib verbosity level. |
| `check_completeness` | `False` | Compatibility argument retained by the API. |
| `dtype` | `np.float32` | Numeric dtype used for node time series and saved FC arrays. |
| `json_path` | `prep_log.json` | Processing-record JSON path updated after FC construction. |
| `overwrite` | `False` | Controls whether matching JSON records are replaced. |
| `log_num` | `00001` | Identifier appended to the progress log filename. |
| `suffix` | `None` | Optional label appended to FC output filenames as `suffix-<suffix>`. |
| `get_name` | `False` | If `True`, returns the saved FC filenames. |

FC output filenames include `desc-fc-zscore` when `zscore=True` and `desc-fc-raw` when `zscore=False`. Common-space and transformation stages should select these files with filters such as `{"desc": "fc-zscore", "suffix": "lowall"}`.

**Cortical**

This example builds cortical seed FC files for `hemi-L` and `hemi-R`, using downsampled cortical target searchlights from both hemispheres.

```python
target_sls_cortical = {
    "hemi-L": sls_surf_target["l"],
    "hemi-R": sls_surf_target["r"],
}

for seed_struct in ["hemi-L", "hemi-R"]:
    ha.fc_build(
        work_dir,
        sub_list,
        seed_step="resample",
        seed_modality="response",
        seed_structure=seed_struct,
        seed_file_filter=fit_response_filter,
        seed_sls=sls_surf_seed,
        target_step="resample",
        target_modality="response",
        target_structure=["hemi-L", "hemi-R"],
        target_file_filter=fit_response_filter,
        target_sls=target_sls_cortical,
        save_modality="connectivity",
        zscore=True,
        njobs=32,
        verbose=1,
        dtype=np.float32,
        json_path=json_path,
        overwrite=False,
        log_num="cha_fc_cortical",
        suffix="lowall",
    )
```

**Subcortical**

This example builds subcortical seed FC files. The target set mixes cortical and subcortical target neighborhoods, following the structure supported by the current GUI pipeline.

```python
target_sls_mixed = {
    "hemi-L": sls_surf_target["l"],
    "hemi-R": sls_surf_target["r"],
    **sls_sub_target,
}

target_structures_mixed = [
    "hemi-L",
    "hemi-R",
    "volume-subcortical-L",
    "volume-subcortical-R",
    "volume-subcortical-BRAIN_STEM",
]

for seed_struct in [
    "volume-subcortical-L",
    "volume-subcortical-R",
    "volume-subcortical-BRAIN_STEM",
]:
    ha.fc_build(
        work_dir,
        sub_list,
        seed_step="resample",
        seed_modality="response",
        seed_structure=seed_struct,
        seed_file_filter=fit_response_filter,
        seed_sls=sls_sub_seed,
        target_step="resample",
        target_modality="response",
        target_structure=target_structures_mixed,
        target_file_filter=fit_response_filter,
        target_sls=target_sls_mixed,
        save_modality="connectivity",
        zscore=True,
        njobs=32,
        verbose=1,
        dtype=np.float32,
        json_path=json_path,
        overwrite=False,
        log_num="cha_fc_subcortical",
        suffix="lowall",
    )
```

**Volume ROI**

This example builds ROI seed FC files with ROI target neighborhoods. The target list can be expanded to include cortical or subcortical targets if the study design requires cross-structure connectivity profiles.

```python
ha.fc_build(
    work_dir,
    sub_list,
    seed_step="resample",
    seed_modality="response",
    seed_structure=f"volume-{roi_name}",
    seed_file_filter=fit_response_filter,
    seed_sls=sls_roi_seed,
    target_step="resample",
    target_modality="response",
    target_structure=[f"volume-{roi_name}"],
    target_file_filter=fit_response_filter,
    target_sls=sls_roi_target,
    save_modality="connectivity",
    zscore=True,
    njobs=32,
    verbose=1,
    dtype=np.float32,
    json_path=json_path,
    overwrite=False,
    log_num="cha_fc_roi",
    suffix="lowall",
)
```

### (3) Common Space Construction

`ha.cspace_sls` is shared with RHA. Its algorithmic arguments have the same meaning as in RHA, including `cspace_kind`, `common_topography`, `weight`, `demean`, `start_sub`, `chunk_size`, `dtype`, and `scope`.

The CHA call differs in the data it reads. `sls`, `dists`, and `radius` describe alignment searchlights over the seed feature axis of FC matrices. `modality` is set to `connectivity`, `step` points to the step where `ha.fc_build` saved FC files, `structure_name` names the seed structures whose FC files were built, and `**file_filters` select FC outputs, commonly `desc="fc-zscore"` plus a `suffix` filter when a suffix was saved. `task_name` should use a CHA label such as `chaPRcortical`, `chaPRsubcortical`, or `chaPRroi`.

**Cortical**

```python
ha.cspace_sls(
    work_dir,
    sls=sls_surf,
    dists=dists_surf,
    radius=radius_surf,
    sub_list=sub_list,
    task_name="chaPRcortical",
    cspace_kind="procrustes",
    common_topography=False,
    weight=True,
    demean=True,
    start_sub=0,
    chunk_size=2000,
    njobs=32,
    step="resample",
    modality="connectivity",
    structure_name=["hemi-L", "hemi-R"],
    dtype=np.float32,
    scope="sls",
    json_path=json_path,
    overwrite=False,
    log_num="cha_cspace_cortical",
    **fc_file_filter,
)
```

**Subcortical**

```python
ha.cspace_sls(
    work_dir,
    sls=sls_sub_align,
    dists=dists_sub_align,
    radius=radius_vol,
    sub_list=sub_list,
    task_name="chaPRsubcortical",
    cspace_kind="procrustes",
    common_topography=False,
    weight=True,
    demean=True,
    start_sub=0,
    chunk_size=2000,
    njobs=32,
    step="resample",
    modality="connectivity",
    structure_name=[
        "volume-subcortical-L",
        "volume-subcortical-R",
        "volume-subcortical-BRAIN_STEM",
    ],
    dtype=np.float32,
    scope="sls",
    json_path=json_path,
    overwrite=False,
    log_num="cha_cspace_subcortical",
    **fc_file_filter,
)
```

**Volume ROI**

```python
ha.cspace_sls(
    work_dir,
    sls=sls_roi_align,
    dists=dists_roi_align,
    radius=radius_vol,
    sub_list=sub_list,
    task_name="chaPRroi",
    cspace_kind="procrustes",
    common_topography=False,
    weight=True,
    demean=True,
    start_sub=0,
    chunk_size=2000,
    njobs=32,
    step="resample",
    modality="connectivity",
    structure_name=[f"volume-{roi_name}"],
    dtype=np.float32,
    scope="sls",
    json_path=json_path,
    overwrite=False,
    log_num="cha_cspace_roi",
    **fc_file_filter,
)
```

### (4) Transformation Matrix Calculation

This stage uses `ha.xform_sls_con`. Its Procrustes and aggregation arguments have the same meaning as in RHA, including `reflection`, `scaling`, `weight`, `chunk_size`, `check_completeness`, `dtype`, `njobs`, and `scope`.

The CHA call estimates sparse transformation matrices by aligning each subject's connectivity files to the CHA common space. Use the same alignment `sls`, `dists`, and `radius` used for CHA common-space construction. Set `source_step` to the step where FC files were saved, set `modality="connectivity"`, select the seed structures in `structure_name`, match `task_name` to the CHA common-space task, and use `cspace_desc` such as `{"alg": "procrustes"}` when needed to identify one common-space file. Source `**file_filters` should select the FC files, commonly `desc="fc-zscore"` plus the FC suffix.

**Cortical**

```python
ha.xform_sls_con(
    work_dir,
    sls=sls_surf,
    dists=dists_surf,
    radius=radius_surf,
    sub_list=sub_list,
    source_step="resample",
    modality="connectivity",
    structure_name=["hemi-L", "hemi-R"],
    task_name="chaPRcortical",
    cspace_desc=common_space_filter,
    reflection=True,
    scaling=False,
    weight=True,
    chunk_size=2000,
    check_completeness=False,
    dtype=np.float32,
    njobs=32,
    scope="sls",
    json_path=json_path,
    overwrite=False,
    log_num="cha_xform_cortical",
    **fc_file_filter,
)
```

**Subcortical**

```python
ha.xform_sls_con(
    work_dir,
    sls=sls_sub_align,
    dists=dists_sub_align,
    radius=radius_vol,
    sub_list=sub_list,
    source_step="resample",
    modality="connectivity",
    structure_name=[
        "volume-subcortical-L",
        "volume-subcortical-R",
        "volume-subcortical-BRAIN_STEM",
    ],
    task_name="chaPRsubcortical",
    cspace_desc=common_space_filter,
    reflection=True,
    scaling=False,
    weight=True,
    chunk_size=2000,
    check_completeness=False,
    dtype=np.float32,
    njobs=32,
    scope="sls",
    json_path=json_path,
    overwrite=False,
    log_num="cha_xform_subcortical",
    **fc_file_filter,
)
```

**Volume ROI**

```python
ha.xform_sls_con(
    work_dir,
    sls=sls_roi_align,
    dists=dists_roi_align,
    radius=radius_vol,
    sub_list=sub_list,
    source_step="resample",
    modality="connectivity",
    structure_name=[f"volume-{roi_name}"],
    task_name="chaPRroi",
    cspace_desc=common_space_filter,
    reflection=True,
    scaling=False,
    weight=True,
    chunk_size=2000,
    check_completeness=False,
    dtype=np.float32,
    njobs=32,
    scope="sls",
    json_path=json_path,
    overwrite=False,
    log_num="cha_xform_roi",
    **fc_file_filter,
)
```

### (5) Apply Transformation Matrix

`ha.align_pipe` is shared with RHA, and its source-file selection and matrix-application behavior stay the same. In a typical CHA workflow, the transformation matrices are estimated from connectivity files, then applied to response data from the same feature space.

The CHA call usually keeps `source_modality="response"` and uses `source_name_filter` to select the response data to align, often a held-out run such as `run="02"`. Set `xform_modality="connectivity"`, because CHA transformation matrices are saved under `xforms/connectivity`; set `xform_structure_name=None` so transform lookup follows the selected source structure; and set `xform_name_filter` to the CHA transform task, such as `{"task": "chaPRcortical"}`.

**Cortical**

```python
for structure in ["hemi-L", "hemi-R"]:
    ha.align_pipe(
        work_dir,
        sub_list,
        source_step="resample",
        source_modality="response",
        source_structure_name=structure,
        source_name_filter=align_source_filter,
        xform_modality="connectivity",
        xform_structure_name=None,
        xform_name_filter={"task": "chaPRcortical"},
        njobs=5,
        verbose=1,
        dtype=np.float32,
        json_path=json_path,
        overwrite=False,
        log_num="cha_align_cortical",
        suffix="run02",
    )
```

**Subcortical**

```python
for structure in [
    "volume-subcortical-L",
    "volume-subcortical-R",
    "volume-subcortical-BRAIN_STEM",
]:
    ha.align_pipe(
        work_dir,
        sub_list,
        source_step="resample",
        source_modality="response",
        source_structure_name=structure,
        source_name_filter=align_source_filter,
        xform_modality="connectivity",
        xform_structure_name=None,
        xform_name_filter={"task": "chaPRsubcortical"},
        njobs=5,
        verbose=1,
        dtype=np.float32,
        json_path=json_path,
        overwrite=False,
        log_num="cha_align_subcortical",
        suffix="run02",
    )
```

**Volume ROI**

```python
ha.align_pipe(
    work_dir,
    sub_list,
    source_step="resample",
    source_modality="response",
    source_structure_name=f"volume-{roi_name}",
    source_name_filter=align_source_filter,
    xform_modality="connectivity",
    xform_structure_name=None,
    xform_name_filter={"task": "chaPRroi"},
    njobs=5,
    verbose=1,
    dtype=np.float32,
    json_path=json_path,
    overwrite=False,
    log_num="cha_align_roi",
    suffix="run02",
)
```

## GUI Pipeline

Launch the graphical interface with `gui()`. After the windows open, configure the relevant panels interactively, enable the stages to run, save or load configurations as needed, and click `RUN!`.

```python
from fmriha.gui import gui

gui()
```

### (1) Searchlight Preparation

The GUI prepares searchlights automatically from `Work Dir`, `n_jobs Parameters`, `Space and Searchlight Configuration`, and the selected structures in the runnable panels. CHA uses the same alignment-searchlight caches as RHA and adds seed and target searchlight roles during `Connectivity Calculation`.

**Screenshot placeholder:** Main GUI after launching `gui()`, with `Work Dir` filled in and the workflow tabs visible.

**Screenshot placeholder:** `Space and Searchlight Configuration` window showing Surface and Volume settings.

**Cortical**

Cortical alignment searchlights follow the same GUI-controlled `nb.sls` call as RHA. `Space and Searchlight Configuration -> Surface -> Space` defaults to `onavg-ico32`, and `Surface Searchlight Radius` defaults to `20`. CHA also uses `Connectivity Calculation -> Target Searchlight Space`, default `onavg-ico8`, as the cortical target `center_space`, and `Target Searchlight Radius (Surface)`, default `13`, as the target-neighborhood radius. Selecting `hemi-L` or `hemi-R` under `Seed Structure` creates single-vertex seeds; selecting them under `Target Structure` creates cortical target neighborhoods.

**Subcortical**

Subcortical alignment searchlights follow the same CIFTI-reference workflow as RHA. `CIFTI Reference` is read from the Volume section, `Volume Searchlight Radius` defaults to `4`, and `Threshold` defaults to `0.2` for dense alignment searchlights. CHA adds target searchlights with `Connectivity Calculation -> Target Searchlight Radius (Volume)`, default `4`; the GUI creates target center masks with `N=1994` and `sls_type="not_seed"`, then uses `threshold=0` for subcortical target neighborhoods. Dense and target searchlights are cached as `masks/sls_dists_vol.pkl` and `masks/sls_sub_dsample.pkl`.

**Volume ROI**

For an ROI, the GUI reads `NIFTI Mask` and `NIFTI ROI Name` from the Volume section. Dense ROI searchlights use `Volume Searchlight Radius`, default `4`, and `Threshold`, default `0.2`, matching the RHA-style volume searchlight call. CHA adds ROI target centers through `downsample_nifti_volume(mask_data)` with its default parameters and caches dense and target ROI searchlights as `masks/sls_dists_vol_<roi_name>.pkl` and `masks/sls_dists_vol_<roi_name>_dsample.pkl`.

### (2) Functional Connectivity Calculation

In the GUI, FC construction is configured in the `Connectivity Calculation` tab. When this panel is checked and `RUN!` is clicked, the GUI runner builds seed and target searchlights, resolves the selected subject lists, and calls `ha.fc_build` once for each selected seed structure and subject-list source.

**Screenshot placeholder:** `Connectivity Calculation` tab with seed fields, target fields, target searchlight settings, `Data List`, and `Suffix` visible.

**Field mapping for `ha.fc_build`**

| Function parameter | Function default | GUI source or automatic value | Meaning |
|---|---|---|---|
| `work_dir` | Required | `Work Dir` | Root HA work directory. |
| `sub_list` | Required | `Data List` selections: Template, Transformation, Alignment, or Other Subjects | Subjects used for FC construction. The GUI runner calls FC construction for each selected list. |
| `seed_step` | Required | `Seed Step`, default `resample` | Step folder containing seed response files. |
| `seed_modality` | Required | `Seed Modality`, default `response` | Modality folder containing seed data. |
| `seed_structure` | Required | `Seed Structure` | One selected seed structure per `ha.fc_build` call. |
| `seed_file_filter` | Required | `Seed File Filter` | BIDS-like filters selecting one seed file per subject. |
| `seed_sls` | Required | Generated from selected seed structures | Single-feature seed definitions for CHA. |
| `target_step` | Required | `Target Step`, default `resample` | Step folder containing target response files. |
| `target_modality` | Required | `Target Modality`, default `response` | Modality folder containing target data. |
| `target_structure` | Required | `Target Structure` | Selected target structures converted to folder names. |
| `target_sls` | Required | Generated from selected target structures and target searchlight settings | Target neighborhoods used to create connectivity profiles. |
| `target_file_filter` | Required | `Target File Filter` | BIDS-like filters selecting one target file per subject and target structure. |
| `save_modality` | `connectivity` | Automatic value `connectivity` | FC output modality folder. |
| `zscore` | `True` | `Zscore`, default enabled | Controls `desc-fc-zscore` or `desc-fc-raw` output. |
| `njobs` | `5` | `n_jobs Parameters -> Connectivity Calculation`, default half of host CPU count | Worker count for FC construction. |
| `verbose` | `1` | Automatic value `1` | Joblib verbosity. |
| `check_completeness` | `False` | Automatic value `False` | Compatibility argument retained by the API. |
| `dtype` | `np.float32` | `Float Type`, default `float32` | Numeric dtype for saved FC arrays. |
| `json_path` | `prep_log.json` | `Work Dir / logs / prep_log.json` | Processing-record JSON path. |
| `overwrite` | `False` | Automatic value `False` | JSON record behavior during GUI runs. |
| `log_num` | `00001` | Automatic value `fc` | Progress log suffix. |
| `suffix` | `None` | `Suffix`, default empty | Optional suffix written to FC output filenames. |
| `get_name` | `False` | Automatic value `False` | FC filenames are saved to disk; the GUI ignores returned filename lists. |

**Cortical**

Select `hemi-L`, `hemi-R`, or both under `Seed Structure`. Select cortical target structures under `Target Structure`. Use file filters that select the response data used for FC construction, such as `run=01` or `set=train`. The resulting FC files are saved under `resample/connectivity/hemi-L` and `resample/connectivity/hemi-R` when `Seed Step` is `resample`.

**Subcortical**

Select one or more of `subcortical-L`, `subcortical-R`, and `subcortical-BRAIN_STEM` under `Seed Structure`. Fill `CIFTI Reference` before running. The GUI maps these names to `volume-subcortical-L`, `volume-subcortical-R`, and `volume-subcortical-BRAIN_STEM` for the function call.

**Volume ROI**

Select `volume-ROI` under `Seed Structure`. Fill `NIFTI Mask` and `NIFTI ROI Name` before running. The GUI maps the selected ROI to `volume-<roi_name>` and saves FC files under that structure folder.

### (3) Common Space Construction

In the GUI, common-space construction is configured in the `Template Calculation` tab. For CHA, the tab should be configured to read the connectivity files produced by `Connectivity Calculation`.

**Screenshot placeholder:** `Template Calculation` tab with the panel enabled, CHA task name entered, `Modality` set to `connectivity`, FC file filters filled, and structures selected.

The GUI maps most `ha.cspace_sls` controls exactly as in RHA: `Common Space Kind`, `Common Topography`, `Weight`, `Demean`, `Start Sub`, `Chunk Size`, `Data Type`, `Scope`, `Sub List`, and the template-calculation worker count keep the same meanings. The GUI also continues to write progress logs with `log_num="cspace"` and records results in `Work Dir / logs / prep_log.json`.

For CHA, set `Template Calculation -> Modality` to `connectivity`, set `Step` to the step where FC files were saved, choose the seed structures whose FC files were built, and use `Task Name` for a CHA label such as `chaPRcortical`. The `File Filter` rows, together with `Suffix` when filled, should select FC outputs such as `desc=fc-zscore` and `suffix=lowall`. Searchlights and distances are generated from the selected structures and should match the FC seed axis.

**Cortical**

Select `hemi-L`, `hemi-R`, or both. Set `Modality` to `connectivity`, choose the step where FC files were saved, and fill file filters that select the FC outputs.

**Subcortical**

Select one or more subcortical structures. The GUI uses the dense CIFTI-derived searchlight cache for common-space construction.

**Volume ROI**

Select `volume-ROI`. The GUI converts it to `volume-<roi_name>` using `NIFTI ROI Name` and uses the dense ROI searchlight cache.

### (4) Transformation Matrix Calculation

In the GUI, transformation matrix calculation is configured in the `Transformation Matrix Calculation` tab. For this CHA version, the GUI run calls `ha.xform_sls_con` for the selected cortical, subcortical, and ROI structures.

**Screenshot placeholder:** `Transformation Matrix Calculation` tab with the panel enabled, CHA task name entered, `Modality` set to `connectivity`, common-space filters visible, and FC source filters filled.

The GUI maps the shared `ha.xform_sls_con` controls as in RHA: `Reflection`, `Scaling`, `Weight`, `Chunk Size`, `Data Type`, `Scope`, `Sub List`, and the transformation worker count keep their meanings. The GUI uses `log_num="xform"`, writes to `Work Dir / logs / prep_log.json`, and passes `check_completeness=False` internally.

For CHA, set `Transformation Matrix Calculation -> Modality` to `connectivity`, set `Step` to the FC output step, select the seed structures whose FC files were built, and match `Task Name` to the CHA common-space task. The `Common Space` rows should identify one CHA common-space file, commonly with `alg=procrustes`. The source `File Filter` rows, together with `Suffix` when filled, should select FC outputs such as `desc=fc-zscore` and `suffix=lowall`.

**Cortical**

Select `hemi-L`, `hemi-R`, or both. Set `Modality` to `connectivity`, match `Task Name` to the cortical CHA common-space task, and use FC source filters.

**Subcortical**

Select one or more subcortical structures. Fill `CIFTI Reference` in `Space and Searchlight Configuration` before running if the searchlight cache is absent.

**Volume ROI**

Select `volume-ROI`. Fill `NIFTI Mask` and `NIFTI ROI Name` before running if the ROI searchlight cache is absent.

### (5) Apply Transformation Matrix

In the GUI, applying transformation matrices is configured in the `Alignment` tab. CHA commonly applies connectivity-derived transformations to response data, so `Source Modality` is usually `response` and `Transform Modality` is `connectivity`.

**Screenshot placeholder:** `Alignment` tab with `Source Modality=response`, `Transform Modality=connectivity`, source filters, transform filters, suffix, and subject list visible.

**Screenshot placeholder:** Bottom action bar showing `Save Configuration`, `Load Configuration`, `Reset All Configuration`, and `RUN!`, plus the sidebar `Monitor` button.

The GUI maps shared `ha.align_pipe` controls as in RHA: `Source Step`, `Source Modality`, `Source Structure`, `Source Name Filter`, `Float Type`, `Suffix`, `Sub List`, and the alignment worker count keep their meanings. The GUI uses `verbose=1`, `overwrite=False`, `log_num="alignment"`, and records the stage in `Work Dir / logs / prep_log.json`.

For CHA, set `Transform Modality` to `connectivity` so the GUI reads transformations from `xforms/connectivity`. The GUI uses the selected source structure for transform lookup and passes the transform filters from `Transform Name Filter`, typically `task=<cha_task_name>`. A common CHA alignment setup uses `Source Modality=response` and applies the connectivity-derived matrices to held-out response data.

**Cortical**

Select `hemi-L`, `hemi-R`, or both as source structures. Use response source filters for the data being aligned, such as `run=02`, and transform filters matching the cortical CHA task.

**Subcortical**

Select one or more subcortical structures. The GUI maps them to `volume-subcortical-L`, `volume-subcortical-R`, and `volume-subcortical-BRAIN_STEM`.

**Volume ROI**

Select `volume-ROI`. The GUI maps it to `volume-<roi_name>` using `NIFTI ROI Name`. Use transform filters matching the ROI CHA task.
